## Domain Model Test

In [ ]:
# %pip install yfinance

In [ ]:
from domain.usd_market_data import USDMarketData

market_data = USDMarketData(
    stock_symbol="NVDA",
    price=180.23,
    volume=80_000_000,
)

## Adapter Interface Test

In [ ]:
from domain.usd_market_data_provider import USDMarketDataProvider
from domain.usd_market_data import USDMarketData


class FakeMarketDataProvider(USDMarketDataProvider):

    def get_market_data(self, stock_symbol: str) -> USDMarketData:
        return USDMarketData(
            stock_symbol=stock_symbol.upper(),
            price=180.25,
            volume=50_000_000,
        )


provider: USDMarketDataProvider = FakeMarketDataProvider()

data = provider.get_market_data("nvda")

data


## Live Yahoo Test

In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

data = provider.get_market_data("NVDA")

data

## Test Analyzer

1. Input domain model
2. MarketAgent.Analyze()
3. Output Domain Model
4. Analysis Result

In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter
from agents.usd_market_agent import USDMarketAgent

provider = USDYahooFinanceAdapter()
agent = USDMarketAgent()

market_data = provider.get_market_data("NVDA")

print(market_data)

result = agent.analyze(market_data)

print(result)

# Register the Market Agent

"NVDA" -> ResearchPlanner -> ResearchTask(task_type="market_analysis", stock_symbol="NVDA")->AgentRouter->AgentRegistry->MarketAgent

In [ ]:
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_research_planner import USDResearchPlanner

# 1. Create registry
registry = USDAgentRegistry()

# 2. Create specialist
market_agent = USDMarketAgent()

# 3. Register specialist
registry.register(
    "market_analysis",
    market_agent
)

# 4. Create router
router = USDAgentRouter(registry)

# 5. Ask router for appropriate agent
agent = router.route("market_analysis")

# agent

planner = USDResearchPlanner()

tasks = planner.plan("NVDA")

task = tasks[0]

# Taking first task
print(task)
print(agent)
print(type(agent))

## Connect Provider Data

Planner driven pipeline
"NVDA" -> ResearchPlanner -> ResearchTask(task_type="market_analysis", stock_symbol="NVDA")->AgentRouter->AgentRegistry->MarketAgent->MarketData->YahooFinanceAdapter->AnalysisResult


In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

# Planner creates tasks
tasks = planner.plan("NVDA")

# Select market task
task = next(
    task
    for task in tasks
    if task.task_type == "market_analysis"
)

# Router selects appropriate specialist
agent = router.route(task.task_type)

# Adapter retrieves normalized market data
market_data = provider.get_market_data(task.stock_symbol)

# Specialist analyzes data
result = agent.analyze(market_data)

print("TASK")
print(task)

print("\nAGENT")
print(type(agent).__name__)

print("\nMARKET DATA")
print(market_data)

print("\nANALYSIS RESULT")
print(result)

## Research Planner

Now the router consumes the task

In [ ]:
from orchestration.usd_research_planner import USDResearchPlanner

planner = USDResearchPlanner()

tasks = planner.plan("NVDA")

print(tasks)

## Connect the Planner to Router


In [ ]:
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from orchestration.usd_research_planner import USDResearchPlanner

researchPlanner = USDResearchPlanner()

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

agent = router.route("market_analysis")

print(agent)
print(type(agent))


## The code will explicitly choose the specialist

NVDA

YahooFinanceAdapter
 
Yahoo Finance
 
Normalization
 
MarketData
 
"market_analysis"
 
AgentRouter
 
AgentRegistry
 
MarketAgent
 
analyze(MarketData)
 
AnalysisResult

In [ ]:
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from orchestration.usd_research_planner import USDResearchPlanner
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

# Get Normalized data
market_data = provider.get_market_data("NVDA")

# Route the Task
agent = router.route("market_analysis")

# Run the Strategy
result = agent.analyze(market_data)

print(result)
print(result.agent)
print(result.summary)
print(result.evidence)

## Test Aggregator

In [ ]:
from domain.usd_analysis_result import USDAnalysisResult
from services.usd_aggregator import USDResultAggregator

market_result = USDAnalysisResult(
    agent="market",
    summary="Market analysis completed.",
    evidence=[
        "price=180.50",
        "volume=1000000"
    ]
)

news_result = USDAnalysisResult(
    agent="news",
    summary="News analysis completed.",
    evidence=[
        "3 recent articles"
    ]
)

results = [
    market_result,
    news_result
]

aggregator = USDResultAggregator()

combined = aggregator.aggregate_results(results)

print(combined)


## Test the Orchestrator

In [ ]:
# from orchestration.usd_research_orchestrator import USDResearchOrchestrator
# from domain.usd_research_task import USDResearchTask
# from domain.usd_market_data import USDMarketData

# class FakeMarketOnlyPlanner:
#     def plan(self, stock_symbol: str):
#         return [
#             USDResearchTask(
#                 stock_symbol=stock_symbol.upper(),
#                 task_type="market_analysis"
#             )
#         ]

# class FakeMarketDataProvider:

#     def get_market_data(self, stock_symbol: str) -> USDMarketData:

#         return USDMarketData(
#             stock_symbol=stock_symbol.upper(),
#             price=180.25,
#             volume=50_000_000
#         )

# from agents.usd_market_agent import USDMarketAgent
# from orchestration.usd_agent_registry import USDAgentRegistry
# from orchestration.usd_agent_router import USDAgentRouter
# from orchestration.usd_research_planner import USDResearchPlanner
# from domain.usd_market_data_provider import USDMarketDataProvider
# registry = USDAgentRegistry()

# registry.register(
#     "market_analysis",
#     USDMarketAgent()
# )

# router = USDAgentRouter(registry)

# print(registry._agents)

# agent = router.route("market_analysis")

# print(type(agent).__name__)

# from services.usd_aggregator import USDResultAggregator

# aggregator = USDResultAggregator()

# # Construct the orchestrator

# planner = FakeMarketOnlyPlanner()
# provider = FakeMarketDataProvider()

# orchestrator = USDResearchOrchestrator(
#     planner=planner,
#     provider=provider,
#     router=router,
#     aggregator=aggregator
# )

# print("Planner:", type(orchestrator.planner).__name__)
# print("Provider:", type(orchestrator.provider).__name__)
# print("Router:", type(orchestrator.router).__name__)
# print("Aggregator:", type(orchestrator.aggregator).__name__)

# # Run the orchestrator
# report = orchestrator.run("NVDA")

# report


## Test the Evaluator Independently

In [ ]:
from domain.usd_analysis_result import USDAnalysisResult
from services.usd_aggregator import USDResultAggregator
from services.usd_evaluator import USDAnalysisEvaluator

# Create Fake Specialist Result
market_result = USDAnalysisResult(
    agent="market",
    summary="NVDA is trading at $180.20.",
    evidence=[
        "price=180.20",
        "volume=2000000"
    ]
)

# Aggregate

aggregator = USDResultAggregator()

combined_report = aggregator.aggregate_results(
    [market_result]
)

# evaluate

evaluator = USDAnalysisEvaluator(
    minimum_specialists=1
)

evaluation = evaluator.evaluate(
    combined_report
)

print(evaluation)
print(evaluation.passed)
print(evaluation.issues)

## Orchestrate with Evaluator

In [ ]:
from orchestration.usd_research_orchestrator import USDResearchOrchestrator
from orchestration.usd_research_planner import USDResearchPlanner

class FakeMarketOnlyPlanner:
    def plan(self, stock_symbol: str):
        return [
            USDResearchTask(
                stock_symbol=stock_symbol.upper(),
                task_type="market_analysis"
            )
        ]

planner = FakeMarketOnlyPlanner()

orchestrator = USDResearchOrchestrator(
    planner=planner,
    provider=provider,
    router=router,
    aggregator=aggregator,
    evaluator=evaluator,
)

report = orchestrator.run(
    "NVDA"
)

print(
    report["state_history"]
)

print(
    report["final_evaluation"]
)

assert report["status"] == "complete"
assert report["final_evaluation"].passed is True
assert report["final_evaluation"].issues == []

report = orchestrator.run("NVDA")


## Test the Optimizer Independently

In [ ]:
from domain.usd_evaluation_result import USDEvaluationResult
from services.usd_optimizer import USDAnalysisOptimizer

## Create failing scenario

evaluation = USDEvaluationResult(
    passed=False,
    issues=[
        "missing_evidence:market"
    ]
)

optimizer = USDAnalysisOptimizer()

actions = optimizer.optimize(
    evaluation
)

print(actions)

## Instantiate the Optimizer


In [ ]:
from domain.usd_market_data_provider import USDMarketDataProvider
from domain.usd_market_data import USDMarketData


class FakeMarketDataProvider(USDMarketDataProvider):

    def get_market_data(self, stock_symbol: str) -> USDMarketData:
        return USDMarketData(
            stock_symbol=stock_symbol.upper(),
            price=180.25,
            volume=50_000_000,
        )


provider: USDMarketDataProvider = FakeMarketDataProvider()

data = provider.get_market_data("nvda")

data

In [ ]:
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_research_planner import USDResearchPlanner

# 1. Create registry
registry = USDAgentRegistry()

# 2. Create specialist
market_agent = USDMarketAgent()

# 3. Register specialist
registry.register(
    "market_analysis",
    market_agent
)

# 4. Create router
router = USDAgentRouter(registry)

# 5. Ask router for appropriate agent
agent = router.route("market_analysis")

# agent

planner = USDResearchPlanner()

tasks = planner.plan("NVDA")

task = tasks[0]

# Taking first task
print(task)
print(agent)
print(type(agent))

## Orchestration Test

Planner -> Provider -> Router -> Agent -> Aggregator -> Evaluator -> Optimizer -> Complete

In [1]:
from domain.usd_market_data import USDMarketData
from domain.usd_research_task import USDResearchTask

from agents.usd_market_agent import USDMarketAgent

from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from orchestration.usd_research_orchestrator import USDResearchOrchestrator

from services.usd_aggregator import USDResultAggregator
from services.usd_analysis_evaluator import USDAnalysisEvaluator
from services.usd_optimizer import USDAnalysisOptimizer

class FakeMarketOnlyPlanner:

    def plan(self, stock_symbol: str):
        return [
            USDResearchTask(
                stock_symbol=stock_symbol.upper(),
                task_type="market_analysis"
            )
        ]

class FakeMarketDataProvider:

    def get_market_data(self, stock_symbol: str) -> USDMarketData:
        return USDMarketData(
            stock_symbol=stock_symbol.upper(),
            price=180.25,
            volume=50_000_000
        )

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

print("REGISTERED AGENTS")
print(registry._agents)

agent = router.route("market_analysis")

assert isinstance(agent, USDMarketAgent)

print("Registry + Router passed")
print("Selected agent:", type(agent).__name__)

planner = FakeMarketOnlyPlanner()

provider = FakeMarketDataProvider()

aggregator = USDResultAggregator()

evaluator = USDAnalysisEvaluator(
    minimum_specialists=1
)

optimizer = USDAnalysisOptimizer()

REGISTERED AGENTS
{'market_analysis': <agents.usd_market_agent.USDMarketAgent object at 0x10a2a2120>}
Registry + Router passed
Selected agent: USDMarketAgent


Test Orchestrator with Retries

In [2]:
from dataclasses import dataclass


class AlwaysFailEvaluator:

    def evaluate(self, report):
        return FakeEvaluationResult(
            passed=False,
            problems=[
                "Insufficient research quality for test"
            ]
        )

@dataclass
class FakeEvaluationResult:
    passed: bool
    problems: list[str]

from dataclasses import dataclass


@dataclass
class FakeEvaluationResult:
    passed: bool
    problems: list[str]

@dataclass
class FakeOptimizationAction:
    action: str
    task_type: str | None = None

class RetryMarketOptimizer:

    def optimize(self, evaluation):
        return [
            FakeOptimizationAction(
                action="request_more_research",
                task_type="market_analysis"
            )
        ]

from domain.usd_market_data import USDMarketData


class CountingFakeMarketDataProvider:

    def __init__(self):
        self.call_count = 0

    def get_market_data(self, stock_symbol: str) -> USDMarketData:

        self.call_count += 1

        return USDMarketData(
            stock_symbol=stock_symbol.upper(),
            price=180.25,
            volume=50_000_000
        )

from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

print(registry._agents)

print(type(router.route("market_analysis")).__name__)

from services.usd_aggregator import USDResultAggregator

aggregator = USDResultAggregator()

planner = FakeMarketOnlyPlanner()

provider = CountingFakeMarketDataProvider()

evaluator = AlwaysFailEvaluator()

optimizer = RetryMarketOptimizer()

from orchestration.usd_research_orchestrator import USDResearchOrchestrator

orchestrator = USDResearchOrchestrator(
    planner=planner,
    provider=provider,
    router=router,
    aggregator=aggregator,
    evaluator=evaluator,
    optimizer=optimizer,
    max_retries=2
)

report = orchestrator.run("NVDA")

report

{'market_analysis': <agents.usd_market_agent.USDMarketAgent object at 0x10a3347d0>}
USDMarketAgent


{'stock_symbol': 'NVDA',
 'status': 'complete',
 'state_history': ['planning',
  'collecting',
  'analyzing',
  'aggregating',
  'evaluating',
  'optimizing',
  'collecting',
  'analyzing',
  'aggregating',
  'evaluating',
  'optimizing',
  'collecting',
  'analyzing',
  'aggregating',
  'evaluating',
  'complete'],
 'combined_report': {'results': [USDAnalysisResult(agent='market', evidence=['volume=50000000', 'price=180.25'], summary='NVDA is trading at value: $180.25.')],
  'specialist_count': 1},
 'the_optimization_actions': [FakeOptimizationAction(action='request_more_research', task_type='market_analysis')],
 'final_evaluation': FakeEvaluationResult(passed=False, problems=['Insufficient research quality for test']),
 'retry_count': 2,
 'max_retries': 2,
 'unresolved_gaps': ['Insufficient research quality for test']}